In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

In [3]:
import numpy as np

asia = df[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))].copy()
asia['year'] = asia['year'].astype(str)      # makes year a category (2 colours), not a gradient

fig = px.scatter(asia, x='gdpPercap', y='lifeExp', color='year',
    color_discrete_map={'2002': '#BBBBBB', '2007': '#0072B2'},
    hover_name='country', log_x=True,
    labels={'gdpPercap': 'GDP per capita (USD, log scale)',
            'lifeExp': 'Life expectancy (years)', 'year': 'Year'},
    title='All 33 Asian countries lived longer in 2007 than in 2002, and 31 also got richer',
    template='simple_white', width=900, height=550)

# arrows from each country's 2002 point to its 2007 point = "highlight the change direction"
w = asia.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
for c in w.index:
    fig.add_annotation(
        x=np.log10(w.loc[c, ('gdpPercap', '2007')]), y=w.loc[c, ('lifeExp', '2007')],
        ax=np.log10(w.loc[c, ('gdpPercap', '2002')]), ay=w.loc[c, ('lifeExp', '2002')],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowwidth=1, arrowcolor='#CCCCCC')

fig.update_traces(marker_size=9)
fig.update_yaxes(range=[35, 85])
fig.show()

In [4]:
d7 = df[df['year'] == 2007].copy()
d7['group'] = d7['continent'].where(d7['continent'] == 'Asia', 'Other continents')

fig = px.scatter(d7, x='gdpPercap', y='lifeExp', size='pop', size_max=60, color='group',
    color_discrete_map={'Asia': '#D55E00', 'Other continents': '#D9D9D9'},
    hover_name='country', log_x=True,
    labels={'gdpPercap': 'GDP per capita (USD, log scale)', 'lifeExp': 'Life expectancy (years)'},
    title='Asia is home to 61% of this dataset\'s population, yet life expectancy ranges from 44 to 83 years',
    template='simple_white', width=950, height=600)

fig.add_annotation(x=np.log10(4959), y=73, text='China and India alone<br>hold over a billion people each',
    showarrow=True, ax=-120, ay=-60, arrowcolor='#D55E00', font=dict(color='#D55E00'))
fig.update_layout(legend_title='')
fig.show()